# Topic 14 - Streaming

Practice consuming SSE streams, assembling deltas, handling mid-stream errors, and understanding the no-retry-after-flush constraint.

## Learning objectives

- parse Server-Sent Events format
- assemble token deltas into complete response
- handle mid-stream errors and `[DONE]` marker
- understand why streaming + tool calls is complex
- explain the no-clean-retry-after-flush constraint

## 1. SSE format basics

Each line starts with `data: `, chunks separated by blank line, ends with `data: [DONE]`

In [ ]:
# Raw SSE stream example (what the server sends)
raw_sse = """data: {"choices":[{"delta":{"content":"Hello"}}]}

data: {"choices":[{"delta":{"content":" world"}}]}

data: {"choices":[{"delta":{"content":"!"}}]}

data: [DONE]
"""

print("Raw SSE:")
print(repr(raw_sse))

## 2. Parse SSE lines

In [ ]:
def parse_sse_lines(text: str):
    """Yield parsed JSON from SSE data lines."""
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if line == "data: [DONE]":
            yield {"done": True}
            continue
        if line.startswith("data: "):
            data = line[6:]  # Remove "data: "
            try:
                yield json.loads(data)
            except json.JSONDecodeError:
                continue

import json

print("Parsed chunks:")
for chunk in parse_sse_lines(raw_sse):
    print(f"  {chunk}")

## 3. Extract content deltas

Streaming responses use `delta.content`, not `message.content`

In [ ]:
def extract_deltas(chunks):
    """Extract content tokens from parsed SSE chunks."""
    for chunk in chunks:
        if chunk.get("done"):
            yield {"type": "done"}
            continue
        
        choices = chunk.get("choices", [])
        if not choices:
            continue
        
        delta = choices[0].get("delta", {})
        
        # Content token
        if "content" in delta and delta["content"]:
            yield {"type": "content", "text": delta["content"]}
        
        # Tool call (preview)
        if "tool_calls" in delta:
            yield {"type": "tool_call", "data": delta["tool_calls"]}
        
        # Finish reason
        if choices[0].get("finish_reason"):
            yield {"type": "finish", "reason": choices[0]["finish_reason"]}

print("Extracted deltas:")
for item in extract_deltas(parse_sse_lines(raw_sse)):
    print(f"  {item}")

## 4. Assemble complete response

Accumulate deltas to reconstruct the full message

In [ ]:
def assemble_response(chunks) -> str:
    """Assemble content deltas into full text."""
    parts = []
    for item in extract_deltas(chunks):
        if item["type"] == "content":
            parts.append(item["text"])
        elif item["type"] == "done":
            break
    return "".join(parts)

full = assemble_response(parse_sse_lines(raw_sse))
print(f"Assembled: {full}")

## 5. Simulated async stream consumer

Practice the async iteration pattern without needing a live server

In [ ]:
# Jupyter already runs an event loop, so use top-level await (not asyncio.run()).
import asyncio
from typing import AsyncGenerator

# Simulated async stream (replace with real aiohttp in production)
async def mock_stream() -> AsyncGenerator[str, None]:
    """Simulates an SSE stream from an LLM API."""
    tokens = ["Hello", " ", "world", "!"]
    for token in tokens:
        await asyncio.sleep(0.05)  # Simulate network delay
        yield f'data: {{"choices":[{{"delta":{{"content":"{token}"}}]}}}}'
    yield 'data: [DONE]'

async def consume_stream():
    """Consume and print tokens as they arrive."""
    full_text = []
    async for line in mock_stream():
        line = line.strip()
        if line == "data: [DONE]":
            break
        if line.startswith("data: "):
            try:
                chunk = json.loads(line[6:])
                delta = chunk["choices"][0]["delta"]
                if "content" in delta:
                    text = delta["content"]
                    full_text.append(text)
                    print(text, end="", flush=True)
            except (json.JSONDecodeError, KeyError):
                continue
    print()
    return "".join(full_text)

result = await consume_stream()
print(f"Final assembled: {result}")

## 6. Mid-stream error handling

Errors can appear after partial content is delivered

In [ ]:
error_sse = """data: {"choices":[{"delta":{"content":"The answer is "}}]}

data: {"choices":[{"delta":{"content":"42"}}]}

data: {"error":{"type":"rate_limit_error","message":"Rate limit exceeded"}}
"""

def consume_with_errors(text: str):
    """Handle errors that appear mid-stream."""
    content_parts = []
    error = None
    
    for chunk in parse_sse_lines(text):
        if chunk.get("done"):
            break
        
        # Check for error in stream
        if "error" in chunk:
            error = chunk["error"]
            break
        
        choices = chunk.get("choices", [])
        if choices and "delta" in choices[0]:
            delta = choices[0]["delta"]
            if "content" in delta:
                content_parts.append(delta["content"])
    
    return "".join(content_parts), error

content, error = consume_with_errors(error_sse)
print(f"Partial content: {content}")
print(f"Error: {error}")

## 7. Tool call streaming (preview)

Tool calls arrive as incremental argument chunks

In [ ]:
tool_call_sse = """data: {"choices":[{"delta":{"tool_calls":[{"index":0,"function":{"name":"get_weather","arguments":"{\"location\"}}]}}]}

data: {"choices":[{"delta":{"tool_calls":[{"index":0,"function":{"arguments":"\"NYC\"}"}}]}}]}

data: {"choices":[{"delta":{"tool_calls":[{"index":0,"function":{"arguments":"}"}}]}}]}

data: [DONE]
"""

def parse_tool_calls(text: str):
    """Assemble tool call arguments from incremental chunks."""
    # Track tool calls by index
    tool_calls = {}
    
    for chunk in parse_sse_lines(text):
        if chunk.get("done"):
            break
        
        choices = chunk.get("choices", [])
        if not choices:
            continue
        
        delta = choices[0].get("delta", {})
        if "tool_calls" not in delta:
            continue
        
        for tc in delta["tool_calls"]:
            idx = tc["index"]
            if idx not in tool_calls:
                tool_calls[idx] = {"name": "", "arguments": ""}
            
            if "name" in tc.get("function", {}):
                tool_calls[idx]["name"] = tc["function"]["name"]
            if "arguments" in tc.get("function", {}):
                tool_calls[idx]["arguments"] += tc["function"]["arguments"]
    
    # Parse complete arguments
    results = []
    for tc in tool_calls.values():
        try:
            args = json.loads(tc["arguments"])
            results.append({"name": tc["name"], "arguments": args})
        except json.JSONDecodeError:
            pass
    
    return results

tool_calls = parse_tool_calls(tool_call_sse)
print(f"Parsed tool calls: {tool_calls}")

## 8. No clean retry after flush — demonstration

In [ ]:
def demonstrate_no_retry():
    """Show why retry after partial output is problematic."""
    
    # Scenario: user asked for a summary
    # Stream starts, we print tokens to console
    # Mid-stream: rate limit error
    
    partial_output = "The quarterly report shows revenue increased by "
    error = "Rate limit exceeded"
    
    print("=== STREAMING SCENARIO ===")
    print(f"Tokens already shown to user: '{partial_output}'")
    print(f"Then error occurs: {error}")
    print()
    print("Options:")
    print("1. Show partial + error message")
    print("2. Buffer entire response first (defeats streaming purpose)")
    print("3. Retry silently (user sees duplicate/conflicting output)")
    print()
    print("CONSTRAINT: Once flushed to user, no clean retry.")

demonstrate_no_retry()

## 9. Streaming vs non-streaming response comparison

In [ ]:
# Non-streaming response structure
non_streaming = {
    "id": "chatcmpl-123",
    "object": "chat.completion",
    "choices": [{
        "index": 0,
        "message": {
            "role": "assistant",
            "content": "Hello world!"
        },
        "finish_reason": "stop"
    }],
    "usage": {"prompt_tokens": 10, "completion_tokens": 5, "total_tokens": 15}
}

# Streaming chunk structure (one of many)
streaming_chunk = {
    "id": "chatcmpl-123",
    "object": "chat.completion.chunk",
    "choices": [{
        "index": 0,
        "delta": {
            "content": "Hello"
        },
        "finish_reason": None
    }]
}

print("Non-streaming content path:", non_streaming["choices"][0]["message"]["content"])
print("Streaming content path:", streaming_chunk["choices"][0]["delta"]["content"])
print()
print("Key difference: message.content vs delta.content")

## 10. Completion exercises

Do these by hand. No autocomplete.

In [ ]:
# EXERCISE 1: Write an async function that streams from a real API
# using aiohttp, yields tokens, and returns the full assembled text

# EXERCISE 2: Modify the consumer to handle the case where
# a tool call arrives mid-stream (buffer arguments until complete)

# EXERCISE 3: Add a timeout per chunk — if no chunk arrives
# within 10 seconds, abort with partial result

# EXERCISE 4: Write a test that simulates a stream that sends
# 3 tokens, then an error, and verify partial content is returned

print("Exercises ready — implement by hand.")